### Testing RAG Applications 📑

#### RAG Application
This application reads data about Model Context Protocol (MCP) server from internet, stores in vector stores, chunks the data with embedding and useful to answer the question about MCP while inferenced.

<img src="./img/RAG.png" width="500" height="400" style="display: block; margin: auto;">

In [1]:
#!pip install -qU langchain-chroma

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# from langchain_ollama import OllamaEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
# from langchain_ollama import ChatOllama

c:\Users\Admin\Documents\GEN_AI\LLM_Evaluation\Test_AI\Dev\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


### If you are using Ollama Model within RAG then perform below step

In [ ]:
# llm = ChatOllama(
#     base_url="http://localhost:11434",
#     model = "qwen2.5:latest",
#     temperature=0.5,
#     max_tokens = 250
# )

### If you are using Groq API Model within RAG then perform below step

In [3]:
# Initialize Groq LLM
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)

In [4]:
# Load data from Web
loader = WebBaseLoader("https://www.descope.com/learn/post/mcp")
data = loader.load()

# Split text into documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(data)

# Add text to vector db
# embedding = OllamaEmbeddings(model="nomic-embed-text:latest")
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

# Create a retriever
retriever = vectordb.as_retriever()

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


template = """Answer the question based only on the following context:

    {context}
    
    Give a summary not the full detail

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)


def retrieve_and_format(question):
    # docs = retriever.get_relevant_documents(question)
    docs = retriever.invoke(question)
    return format_docs(docs)

chain = {"context": retrieve_and_format, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()


C:\Users\Admin\AppData\Local\Temp\ipykernel_30752\3881388503.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


#### Output of the LLM Application

In [5]:
response = chain.invoke("What is MCP")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [6]:
print(response)

MCP (Model Context Protocol) is a protocol that enables communication between AI applications and servers, allowing for the exposure of specific functions and capabilities. It acts as a bridge between developer tooling and consumer use cases, facilitating connections and data exchange.


### Testing RAG Application with DeepEval
<img src="./img/RAGTesting.png" width="800" height="400" style="display: block; margin: auto;">

In [ ]:
### If you are not using Ollama Provider for Deepeval then run the below command to unset Ollama


# !deepeval unset-ollama


🙌 OpenAI will still be used by default because OPENAI_API_KEY is set.


In [8]:
### Login to Confident AI 

import deepeval
deepeval.login(os.getenv("DEEPEVAL_API_KEY"))

🎉🥳 Congratulations! You've successfully logged in! 🙌

In [ ]:
### if you want to use Groq API as you LLM provider for Deepeval then please perform the below configuation

import os
os.environ["OPENAI_BASE_URL"] = ""  # Groq’s OpenAI-compatible API
os.environ["OPENAI_API_KEY"] = ""   # use your Groq key here

In [ ]:
### Set Groq model as local model in Deepeval

!deepeval set-local-model --model-name="openai/gpt-oss-20b" --base-url="" --api-key=""

Settings updated for this session. To persist, use --save=dotenv[:path] 
(default .env.local) or set DEEPEVAL_DEFAULT_SAVE=dotenv:.env.local
🙌 Congratulations! You're now using a local model `openai/gpt-oss-20b` for all
evals that require an LLM.


In [ ]:
### Create Test Case and Evaluation Dataset in Deepeval

from deepeval.test_case import LLMTestCase
from deepeval.dataset import EvaluationDataset

test_case = LLMTestCase(
    input="What is MCP?",
    actual_output=chain.invoke("What is MCP"),
    expected_output="The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps"
)

dataset = EvaluationDataset()
dataset.add_test_case(test_case=test_case)

In [12]:
dataset

EvaluationDataset(test_cases=[LLMTestCase(input='What is MCP?', actual_output='MCP (Model Context Protocol) is a protocol that enables communication between AI applications and servers, allowing for the exposure of specific functions and capabilities. It acts as a bridge between developer tooling and consumer use cases, facilitating connections and data exchange.', expected_output='The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps', context=None, retrieval_context=None, additional_metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None)], goldens=[], _alias=None, _id=None, _multi_turn=False)

In [21]:
dataset.test_cases

[LLMTestCase(input='What is MCP?', actual_output='MCP (Model Context Protocol) is a protocol that enables communication between AI applications and servers, allowing for the exposure of specific functions and capabilities. It acts as a bridge between developer tooling and consumer use cases, facilitating connections and data exchange.', expected_output='The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps', context=None, retrieval_context=None, additional_metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None)]

In [13]:
### Define Custom Evaluation Metric (Concise) using GEval


from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics import GEval

concise_metrics = GEval(
    name = "Concise",
    criteria="Assess if the actual output remains concise while preserving all essential information.",
    
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ]
)

In [14]:
### Define Custom Evaluation Metric (Completeness) using GEval

from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics import GEval

completness_metrics = GEval(
    name = "Completeness",
    criteria="Assess whether the actual output retains all the key information from the input",
    
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ]
)

### Evaluation with GEval 

In [ ]:
### Evaluate Test Case using Built-in and Custom Metrics

from deepeval.evaluate import evaluate
from deepeval.metrics import AnswerRelevancyMetric

evaluate(dataset.test_cases, metrics=[completness_metrics, AnswerRelevancyMetric(),concise_metrics])

✨ You're running DeepEval's latest Completeness [GEval] Metric! (using openai/gpt-oss-20b (Local Model), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using openai/gpt-oss-20b (Local Model), strict=False,
async_mode=True)...

✨ You're running DeepEval's latest Concise [GEval] Metric! (using openai/gpt-oss-20b (Local Model), strict=False, 
async_mode=True)...



Metrics Summary

  - ❌ Completeness [GEval] (score: 0.0, threshold: 0.5, strict: False, evaluation model: openai/gpt-oss-20b (Local Model), reason: The input provides only the actual output with no expected reference text or key information items, making it impossible to verify alignment or completeness according to the evaluation steps., error: None)
  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: openai/gpt-oss-20b (Local Model), reason: The score is 1.00 because the response correctly and completely answered the question without any irrelevant content., error: None)
  - ❌ Concise [GEval] (score: 0.4, threshold: 0.5, strict: False, evaluation model: openai/gpt-oss-20b (Local Model), reason: The output succinctly covers the core purpose of MCP and its role in bridging AI applications with servers, meeting conciseness requirements. However, it omits potential critical details such as specific use cases, technical specifications, or examples that m

⚠ WARNING: No hyperparameters logged.
» ]8;id=869429;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=373225;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhgviwqf01xpmu0g3v8hltd3/regression-testing\https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhgviwqf01xpmu0g3v8hltd3/regression-testi]8;;\
]8;id=373225;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhgviwqf01xpmu0g3v8hltd3/regression-testing\ng]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Completeness [GEval]', threshold=0.5, success=False, score=0.0, reason='The input provides only the actual output with no expected reference text or key information items, making it impossible to verify alignment or completeness according to the evaluation steps.', strict_mode=False, evaluation_model='openai/gpt-oss-20b (Local Model)', error=None, evaluation_cost=0.0, verbose_logs='Criteria:\nAssess whether the actual output retains all the key information from the input \n \nEvaluation Steps:\n[\n    "Identify all key information items in the input.",\n    "Locate corresponding information in the actual output.",\n    "Verify that each key item is present and accurately represented.",\n    "Confirm that no key information is omitted or incorrectly altered."\n] \n \nRubric:\nNone \n \nScore: 0.0'), MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason